# Data Loader

> Load and parse `data/data.json` from greeny/SatisfactoryTools into clean Python dicts.
>
> Exports: `load_data`, `get_item_recipe`

In [ ]:
#| default_exp data

In [ ]:
#| export
import json
from pathlib import Path

## Load raw data

In [ ]:
#| export
def load_data(data_path: str | Path = None) -> dict:
    """Load data.json and return the raw dict with keys: items, recipes, buildings, etc.
    
    If data_path is not provided, resolves relative to this file's location:
    walks up to find the repo root (where data/data.json lives).
    """
    if data_path is None:
        # Walk up from this file to find data/data.json
        here = Path(__file__).resolve().parent
        # Try up to 3 levels up
        for _ in range(4):
            candidate = here / 'data' / 'data.json'
            if candidate.exists():
                data_path = candidate
                break
            here = here.parent
        else:
            raise FileNotFoundError('data/data.json not found. Pass data_path explicitly.')
    with open(data_path, encoding='utf-8') as f:
        return json.load(f)

## Wiki image URL helper

In [ ]:
#| export
def wiki_image_url(item_name: str, size: int = 40) -> str:
    """Return the wiki.gg thumbnail URL for an item by display name.
    
    Pattern: https://satisfactory.wiki.gg/images/thumb/{Name}.png/{size}px-{Name}.png
    Spaces are replaced with underscores.
    """
    slug = item_name.replace(' ', '_')
    return f'https://satisfactory.wiki.gg/images/thumb/{slug}.png/{size}px-{slug}.png'

## Item recipe lookup

In [ ]:
#| export
def get_item_recipe(item_name: str, data: dict, alternate: bool = False) -> dict | None:
    """Look up an item by display name and return a clean recipe dict.

    Returns a dict with:
        name          -- item display name
        description   -- item description
        image_url     -- wiki.gg thumbnail URL
        ingredients   -- list of {name, amount, rate_per_min}
        products      -- list of {name, amount, rate_per_min}
        machine       -- building display name
        machine_power -- power draw in MW
        cycle_time    -- seconds per cycle
        recipe_name   -- recipe display name
        alternate     -- True if this is an alternate recipe

    If alternate=False (default), returns the standard (non-alternate) recipe.
    If alternate=True, returns the first alternate recipe found.
    Returns None if no matching item or recipe is found.
    """
    items = data.get('items', {})
    recipes = data.get('recipes', {})
    buildings = data.get('buildings', {})

    # Find item class key by display name
    item_key = None
    item_data = None
    for key, item in items.items():
        if item.get('name', '').lower() == item_name.lower():
            item_key = key
            item_data = item
            break
    if item_key is None:
        return None

    # Find matching recipe (standard or alternate)
    recipe_data = None
    for recipe in recipes.values():
        if not recipe.get('inMachine', False):
            continue
        products = recipe.get('products', [])
        if not any(p['item'] == item_key for p in products):
            continue
        is_alt = recipe.get('alternate', False)
        if is_alt == alternate:
            recipe_data = recipe
            break

    if recipe_data is None:
        return None

    cycle_time = recipe_data.get('time', 1)  # seconds
    cycles_per_min = 60.0 / cycle_time

    def resolve_ingredient(entry):
        """Resolve an ingredient/product entry to {name, amount, rate_per_min}."""
        ref_key = entry['item']
        amount = entry['amount']
        # Look up in items first, then resources
        ref_item = items.get(ref_key) or data.get('resources', {}).get(ref_key, {})
        return {
            'name': ref_item.get('name', ref_key),
            'amount': amount,
            'rate_per_min': round(amount * cycles_per_min, 4),
        }

    ingredients = [resolve_ingredient(e) for e in recipe_data.get('ingredients', [])]
    products = [resolve_ingredient(e) for e in recipe_data.get('products', [])]

    # Resolve machine
    produced_in = recipe_data.get('producedIn', [])
    machine_key = produced_in[0] if produced_in else None
    machine_data = buildings.get(machine_key, {}) if machine_key else {}
    machine_name = machine_data.get('name', machine_key or 'Unknown')
    machine_power = machine_data.get('metadata', {}).get('powerConsumption', 0)

    return {
        'name': item_data.get('name', item_name),
        'description': item_data.get('description', ''),
        'image_url': wiki_image_url(item_data.get('name', item_name)),
        'ingredients': ingredients,
        'products': products,
        'machine': machine_name,
        'machine_power': machine_power,
        'cycle_time': cycle_time,
        'recipe_name': recipe_data.get('name', ''),
        'alternate': recipe_data.get('alternate', False),
    }

## Test: all 6 target items

In [ ]:
# Not exported — test only
from pathlib import Path

DATA_PATH = Path('../data/data.json')
data = load_data(DATA_PATH)

targets = [
    'Versatile Framework',
    'Modular Engine',
    'Adaptive Control Unit',
    'Heavy Modular Frame',
    'Automated Wiring',
    'Cable',
]

for name in targets:
    result = get_item_recipe(name, data)
    if result is None:
        print(f'FAIL: {name} — not found')
    else:
        ing_str = ', '.join(f"{i['name']} x{i['amount']}" for i in result['ingredients'])
        prod = result['products'][0]
        print(f"OK: {result['name']}")
        print(f"    Recipe:      {result['recipe_name']}")
        print(f"    Machine:     {result['machine']} ({result['machine_power']} MW)")
        print(f"    Cycle:       {result['cycle_time']}s")
        print(f"    Output:      {prod['amount']} @ {prod['rate_per_min']}/min")
        print(f"    Ingredients: {ing_str}")
        print(f"    Image URL:   {result['image_url']}")
        print()